In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


In [1]:
from pyspark.sql import functions as F

level_events = spark.table(
    "lh_silver_game.level_events_clean"
)

StatementMeta(, afe4fda7-b33a-488c-9771-7b3a3cebcea9, 3, Finished, Available, Finished, False)

In [2]:
level_performance = (
    level_events
    .groupBy("level_number")
    .agg(
        F.sum(
            F.when(F.col("event_name") == "level_start", 1).otherwise(0)
        ).alias("level_starts"),

        F.sum(
            F.when(F.col("event_name") == "level_complete", 1).otherwise(0)
        ).alias("level_completes"),

        F.sum(
            F.when(F.col("event_name") == "level_fail", 1).otherwise(0)
        ).alias("level_fails"),

        F.avg("attempt_number").alias("avg_attempt_number"),

        F.sum(
            F.when(F.col("booster_used") == True, 1).otherwise(0)
        ).alias("booster_usage_count"),

        F.avg("moves_remaining").alias("avg_moves_remaining")
    )
)

StatementMeta(, afe4fda7-b33a-488c-9771-7b3a3cebcea9, 4, Finished, Available, Finished, False)

In [3]:
level_performance = (
    level_performance
    .withColumn(
        "completion_rate",
        F.when(
            (F.col("level_completes") + F.col("level_fails")) > 0,
            F.col("level_completes") /
            (F.col("level_completes") + F.col("level_fails"))
        ).otherwise(0)
    )
    .withColumn(
        "fail_rate",
        F.when(
            (F.col("level_completes") + F.col("level_fails")) > 0,
            F.col("level_fails") /
            (F.col("level_completes") + F.col("level_fails"))
        ).otherwise(0)
    )
)

StatementMeta(, afe4fda7-b33a-488c-9771-7b3a3cebcea9, 5, Finished, Available, Finished, False)

In [4]:
print("Level count:", level_performance.count())

display(
    level_performance
    .orderBy("level_number")
    .limit(20)
)

StatementMeta(, afe4fda7-b33a-488c-9771-7b3a3cebcea9, 6, Finished, Available, Finished, False)

Level count: 459


SynapseWidget(Synapse.DataFrame, 8fb88658-f26e-40bb-abee-64c963176be8)

In [5]:
level_performance = (
    level_performance
    .withColumn(
        "booster_usage_rate",
        F.when(
            (F.col("level_starts") + F.col("level_completes") + F.col("level_fails")) > 0,
            F.col("booster_usage_count") /
            (F.col("level_starts") + F.col("level_completes") + F.col("level_fails"))
        ).otherwise(0)
    )
)

StatementMeta(, afe4fda7-b33a-488c-9771-7b3a3cebcea9, 7, Finished, Available, Finished, False)

In [6]:
display(
    level_performance
    .select(
        "level_number",
        "level_starts",
        "level_completes",
        "level_fails",
        "completion_rate",
        "fail_rate",
        "avg_attempt_number",
        "booster_usage_count",
        "booster_usage_rate",
        "avg_moves_remaining"
    )
    .orderBy("level_number")
    .limit(20)
)

StatementMeta(, afe4fda7-b33a-488c-9771-7b3a3cebcea9, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4548c691-e1c0-44a9-8482-15658c67358b)

In [7]:
level_performance.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("lh_gold_game.level_performance")

StatementMeta(, afe4fda7-b33a-488c-9771-7b3a3cebcea9, 9, Finished, Available, Finished, False)

In [8]:
df_check = spark.table("lh_gold_game.level_performance")

print("Saved row count:", df_check.count())

StatementMeta(, afe4fda7-b33a-488c-9771-7b3a3cebcea9, 10, Finished, Available, Finished, False)

Saved row count: 459
